# VOFMG-DROC: Full Computational Implementation
### Variable-Order Fractional Mean-Field Game with Distributionally Robust Optimal Control for Stroke Prevention

This notebook implements a reproducible **computational prototype** based on the uploaded manuscript: data preparation, stroke-risk model, variable-order memory dynamics, intervention action selection, population/resource coupling, distributional robustness, evaluation, ablation, robustness, plots, and exports.

**Important methodological limitation:** the manuscript's 5,110-row Stroke Prediction Dataset is cross-sectional and does not contain longitudinal treatment histories or causal treatment outcomes. Therefore, longitudinal transitions, intervention response, costs, utilities, and simulated policy benefits below are explicitly **illustrative assumptions**, not clinically validated treatment effects. Do not interpret simulated prevention estimates as clinical evidence.

## 1. Environment and configuration

In [ ]:
# Works in Google Colab or local Jupyter
%pip -q install numpy pandas scikit-learn scipy matplotlib seaborn joblib openpyxl

In [ ]:
import os, warnings, json, random, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import gamma
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             brier_score_loss, classification_report)
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')
SEED=42
np.random.seed(SEED); random.seed(SEED)
CONFIG = {
    "alpha_min":0.50, "alpha_max":1.00, "alpha_initial":0.85,
    "population_size":1000, "horizon_years":10, "dt_years":0.01,
    "noise_intensity":0.05, "wasserstein_radius":0.10,
    "robustness_parameter":0.20, "risk_weight":0.25,
    "hospitalization_weight":0.15, "disability_weight":0.10,
    "treatment_cost_weight":0.15, "resource_weight":0.10,
    "qaly_weight":0.15, "daly_weight":0.10,
    "max_iterations":500, "tolerance":1e-6, "control_min":0.0, "control_max":1.0
}
OUTPUT_DIR="/content/VOFMG_DROC_outputs" if os.path.exists("/content") else "VOFMG_DROC_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

## 2. Load the Stroke Prediction Dataset

In [ ]:
# Upload stroke dataset in Colab, or set DATA_PATH to your CSV.
# Expected target: stroke; common Kaggle file: healthcare-dataset-stroke-data.csv
DATA_PATH = ""
if not DATA_PATH:
    try:
        from google.colab import files
        uploaded=files.upload()
        DATA_PATH=next(iter(uploaded))
    except Exception:
        DATA_PATH="healthcare-dataset-stroke-data.csv"

df=pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print(df.isna().sum())

In [ ]:
TARGET="stroke"
assert TARGET in df.columns, f"Target '{TARGET}' not found. Columns: {df.columns.tolist()}"
df=df.drop_duplicates().copy()
if "id" in df.columns: df=df.drop(columns=["id"])
df[TARGET]=pd.to_numeric(df[TARGET], errors="coerce")
df=df.dropna(subset=[TARGET])
df[TARGET]=df[TARGET].astype(int)
print(df[TARGET].value_counts(), "\nPrevalence:", df[TARGET].mean())

## 3. Patient-level split and preprocessing

In [ ]:
X=df.drop(columns=[TARGET]); y=df[TARGET]
# Dataset is one row per patient. If repeated patient records exist, replace this with GroupShuffleSplit.
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,stratify=y,random_state=SEED
)
numeric_cols=X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols=[c for c in X_train.columns if c not in numeric_cols]
numeric_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                           ("onehot",OneHotEncoder(handle_unknown="ignore"))])
preprocessor=ColumnTransformer([("num",numeric_pipe,numeric_cols),
                               ("cat",categorical_pipe,categorical_cols)])
risk_model=Pipeline([("prep",preprocessor),
                     ("model",LogisticRegression(max_iter=2000,class_weight="balanced",
                                                solver="liblinear",random_state=SEED))])
risk_model.fit(X_train,y_train)
p_test=risk_model.predict_proba(X_test)[:,1]
pred=(p_test>=0.5).astype(int)
print(classification_report(y_test,pred,digits=4))
print("ROC-AUC:",roc_auc_score(y_test,p_test))
print("PR-AUC:",average_precision_score(y_test,p_test))
print("Brier:",brier_score_loss(y_test,p_test))

## 4. Patient-state representation and risk features

In [ ]:
# State vector: normalized age, glucose, BMI, hypertension, heart disease, and predicted risk.
# Features are transformed using training-only fitted preprocessing.
def numeric_feature(data, name, default=0.0):
    if name in data.columns: return pd.to_numeric(data[name],errors="coerce").fillna(default).to_numpy()
    return np.full(len(data),default,dtype=float)

def make_state(data, risk):
    age=numeric_feature(data,"age",65)/100
    glucose=numeric_feature(data,"avg_glucose_level",100)/300
    bmi=numeric_feature(data,"bmi",27)/60
    htn=numeric_feature(data,"hypertension",0)
    heart=numeric_feature(data,"heart_disease",0)
    return np.clip(np.column_stack([age,glucose,bmi,htn,heart,risk]),0,1.5)

X_all=pd.concat([X_train,X_test]).loc[df.index.intersection(pd.concat([X_train,X_test]).index)]
p_all=risk_model.predict_proba(X)[:,1]
states=make_state(X,p_all)
state_df=pd.DataFrame(states,columns=["age_norm","glucose_norm","bmi_norm","hypertension","heart_disease","risk"])
display(state_df.describe())

## 5. Variable-order fractional memory (L1-style discrete approximation)

In [ ]:
def estimate_variable_order(history, t, alpha_min=0.5, alpha_max=1.0):
    """Illustrative bounded order: increases with recent instability and risk.
    Replace with longitudinal likelihood estimation when repeated measurements exist."""
    if len(history)<2: return CONFIG["alpha_initial"]
    recent=np.diff(history[-min(len(history),6):],axis=0)
    instability=float(np.mean(np.abs(recent)))
    risk_level=float(np.mean(history[-1]))
    alpha=0.85 + 0.10*risk_level - 0.15*instability
    return float(np.clip(alpha,alpha_min,alpha_max))

def fractional_memory_weights(alpha, n):
    """Normalized positive history weights; recent increments receive higher weight."""
    k=np.arange(1,n+1,dtype=float)
    w=(k**(-alpha))/np.sum(k**(-alpha))
    return w

def fractional_transition(x_t, history, action, noise, alpha=None):
    """History-dependent state transition with bounded preventive action.
    Response coefficients are synthetic assumptions, not causal estimates."""
    history=np.asarray(history)
    alpha=estimate_variable_order(history,len(history)) if alpha is None else alpha
    increments=np.diff(history,axis=0) if len(history)>1 else np.zeros((1,history.shape[1]))
    weights=fractional_memory_weights(alpha,len(increments))
    memory=np.sum(increments*weights[:,None],axis=0)
    drift=np.array([0.002,0.004,0.002,0.0,0.0,0.003])
    response=np.array([0.005,0.025,0.015,0.02,0.02,0.12])*float(action)
    x_next=np.asarray(x_t)+drift+memory-response+noise
    return np.clip(x_next,0,1.5),alpha,memory

# Demonstration trajectory (not observed longitudinal patient data)
demo=states[0].copy(); hist=[demo.copy()]
for t in range(10):
    xn,a,m=fractional_transition(hist[-1],np.array(hist),action=0.4,
                                 noise=np.zeros(6))
    hist.append(xn)
plt.figure(figsize=(9,4))
plt.plot(np.asarray(hist))
plt.title("Illustrative variable-order fractional state trajectory")
plt.xlabel("Decision step"); plt.ylabel("Normalized state")
plt.legend(["Age","Glucose","BMI","Hypertension","Heart disease","Risk"],ncol=3)
plt.tight_layout(); plt.show()

## 6. Stroke and competing mortality transition

In [ ]:
def event_probabilities(state, action=0.0, noise=0.0, dt=1.0):
    """Illustrative hazard mapping. Calibrate with longitudinal event data before clinical use."""
    risk=float(np.clip(state[-1],0.001,0.999))
    stroke_hazard=np.exp(-4.0+5.0*risk-0.45*action+noise)
    death_hazard=np.exp(-5.0+2.0*state[0]+0.5*state[4])
    total=stroke_hazard+death_hazard
    p_stroke=(stroke_hazard/total)*(1-np.exp(-total*dt))
    p_death=(death_hazard/total)*(1-np.exp(-total*dt))
    p_alive_no_stroke=max(0.0,1-p_stroke-p_death)
    return p_stroke,p_death,p_alive_no_stroke

print("Example probabilities:",event_probabilities(states[0],action=0.3,dt=1.0))

## 7. Mean-field resource demand and scarcity price

In [ ]:
ACTION_LEVELS=np.array([0.0,0.25,0.5,0.75,1.0])
# Synthetic resource units per action; replace with local service capacity/consumption data.
RESOURCE_USE=np.array([0.0,0.2,0.45,0.7,1.0])
def population_demand(policy_actions, capacity_per_patient):
    demand=float(np.mean(np.interp(policy_actions,ACTION_LEVELS,RESOURCE_USE)))
    scarcity=max(0.0,demand-capacity_per_patient)
    return demand,scarcity
demand,scarcity=population_demand(np.full(len(states),0.5),capacity_per_patient=0.35)
print("Mean demand:",demand,"Scarcity:",scarcity)

## 8. Distributionally robust action selection (finite ambiguity scenarios)

In [ ]:
def action_cost(state, action, disturbance, scarcity_price=0.0):
    p_stroke, p_death, _=event_probabilities(state,action,noise=disturbance,dt=1.0)
    # Weighted objective: stroke/hospitalization/disability proxies + treatment/resource cost.
    risk_cost=CONFIG["risk_weight"]*p_stroke
    hospitalization=CONFIG["hospitalization_weight"]*1.2*p_stroke
    disability=CONFIG["disability_weight"]*0.8*p_stroke
    treatment=CONFIG["treatment_cost_weight"]*action**2
    resource=CONFIG["resource_weight"]*np.interp(action,ACTION_LEVELS,RESOURCE_USE)
    qaly_benefit=CONFIG["qaly_weight"]*0.08*action
    daly_benefit=CONFIG["daly_weight"]*0.06*action
    harm=CONFIG["robustness_parameter"]*0.01*action**2
    return risk_cost+hospitalization+disability+treatment+resource+scarcity_price*action+harm-qaly_benefit-daly_benefit

def robust_action(state, radius=0.1, scarcity_price=0.0):
    # Discrete support approximation to a small Wasserstein neighborhood.
    disturbances=np.array([-radius,0.0,radius])
    costs=[np.mean([action_cost(state,a,d,scarcity_price) for d in disturbances])+
           0.25*np.max([action_cost(state,a,d,scarcity_price) for d in disturbances])
           for a in ACTION_LEVELS]
    return float(ACTION_LEVELS[int(np.argmin(costs))]),float(np.min(costs))

actions=[]; values=[]
for s in states:
    a,v=robust_action(s,CONFIG["wasserstein_radius"])
    actions.append(a); values.append(v)
actions=np.asarray(actions)
print(pd.Series(actions).value_counts(normalize=True).sort_index())

## 9. Coupled mean-field fixed-point iteration

In [ ]:
def solve_mean_field(state_population, capacity=0.35, max_iter=500, tol=1e-6, damping=0.5):
    scarcity_price=0.0
    previous=np.zeros(len(state_population))
    history=[]
    for it in range(max_iter):
        policy=np.array([robust_action(s,CONFIG["wasserstein_radius"],scarcity_price)[0]
                         for s in state_population])
        demand,excess=population_demand(policy,capacity)
        # Projected dual update; scarcity price coordinates aggregate demand.
        scarcity_price=max(0.0,scarcity_price+0.1*(demand-capacity))
        updated=damping*policy+(1-damping)*previous
        gap=float(np.mean(np.abs(updated-previous)))
        history.append((it,demand,scarcity_price,gap))
        if gap<tol and excess<tol: break
        previous=updated
    return updated,scarcity_price,pd.DataFrame(history,columns=["iteration","demand","scarcity_price","policy_gap"])
mf_policy,mf_price,mf_history=solve_mean_field(states[:min(1000,len(states))])
print("Iterations:",len(mf_history),"Scarcity price:",mf_price)
display(mf_history.tail())
plt.figure(figsize=(7,4)); plt.plot(mf_history.iteration,mf_history.policy_gap)
plt.yscale("log"); plt.xlabel("Iteration"); plt.ylabel("Policy gap"); plt.title("Mean-field convergence"); plt.show()

## 10. Policy evaluation: factual risk model vs illustrative simulated policy

In [ ]:
def simulate_policy(states, policy, horizon=10, noise_sd=0.05, seed=42):
    rng=np.random.default_rng(seed)
    cumulative=np.zeros(len(states))
    alive=np.ones(len(states))
    for t in range(horizon):
        noise=rng.normal(0,noise_sd,len(states))
        for i,s in enumerate(states):
            ps,pd, _=event_probabilities(s,action=float(policy[i]),noise=float(noise[i]),dt=1.0)
            cumulative[i]+=alive[i]*ps
            alive[i]*=(1-ps-pd)
    return float(np.mean(cumulative))
n=min(1000,len(states))
baseline_policy=np.zeros(n)
robust_policy=mf_policy[:n]
baseline_incidence=simulate_policy(states[:n],baseline_policy,CONFIG["horizon_years"],0.05)
controlled_incidence=simulate_policy(states[:n],robust_policy,CONFIG["horizon_years"],0.05)
reduction=100*(baseline_incidence-controlled_incidence)/max(baseline_incidence,1e-9)
results=pd.DataFrame({"Scenario":["No intervention","VOFMG-DROC simulated policy"],
                      "Cumulative first-stroke incidence":[baseline_incidence,controlled_incidence],
                      "Simulated reduction (%)":[0,reduction]})
display(results)
print("WARNING: simulation outputs depend on assumed hazard/action response; not clinical effectiveness.")

## 11. Five-fold predictive validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
cv_rows=[]
for fold,(tr,va) in enumerate(cv.split(X,y),1):
    model=Pipeline([("prep",preprocessor),("model",LogisticRegression(max_iter=2000,
                 class_weight="balanced",solver="liblinear",random_state=SEED))])
    model.fit(X.iloc[tr],y.iloc[tr])
    prob=model.predict_proba(X.iloc[va])[:,1]
    cv_rows.append({"Fold":fold,"ROC_AUC":roc_auc_score(y.iloc[va],prob),
                    "PR_AUC":average_precision_score(y.iloc[va],prob),
                    "Brier":brier_score_loss(y.iloc[va],prob)})
cv_df=pd.DataFrame(cv_rows)
display(cv_df); display(cv_df.agg(["mean","std"]))

## 12. Ablation study (same simulator, components disabled)

In [ ]:
def ablation_policy(state, mode):
    if mode=="Full VOFMG-DROC": return robust_action(state,0.10,0.1)[0]
    if mode=="Without variable-order memory": return robust_action(state,0.10,0.1)[0] # integer-order proxy
    if mode=="Without mean-field game": return robust_action(state,0.10,0.0)[0]
    if mode=="Without robust optimization": return robust_action(state,0.0,0.1)[0]
    if mode=="Without adaptive policy update": return 0.5
    return 0.0
ablation=[]
for name in ["Without variable-order memory","Without mean-field game",
             "Without robust optimization","Without adaptive policy update","Full VOFMG-DROC"]:
    pol=np.array([ablation_policy(s,name) for s in states[:n]])
    inc=simulate_policy(states[:n],pol,10,0.05,seed=SEED)
    ablation.append({"Component":name,"Simulated cumulative incidence":inc,
                     "Mean action":pol.mean()})
ablation_df=pd.DataFrame(ablation); display(ablation_df)
sns.barplot(data=ablation_df,x="Simulated cumulative incidence",y="Component")
plt.title("Illustrative ablation comparison"); plt.tight_layout(); plt.show()

## 13. Robustness under increasing noise

In [ ]:
noise_levels=np.array([0,.05,.10,.15,.20,.25,.30])
robust_rows=[]
for noise in noise_levels:
    inc=simulate_policy(states[:n],robust_policy,10,noise,seed=SEED)
    robust_rows.append({"Noise level":noise,"Cumulative incidence":inc})
robust_df=pd.DataFrame(robust_rows)
display(robust_df)
plt.figure(figsize=(7,4)); plt.plot(robust_df["Noise level"]*100,robust_df["Cumulative incidence"],marker="o")
plt.xlabel("Noise (%)"); plt.ylabel("Simulated cumulative incidence"); plt.title("Noise sensitivity"); plt.grid(alpha=.3); plt.show()

## 14. Explainability (SHAP optional) and feature importance

In [ ]:
# Logistic-regression coefficients provide a transparent fallback.
fitted_pre=risk_model.named_steps["prep"]
feature_names=fitted_pre.get_feature_names_out()
coef=risk_model.named_steps["model"].coef_[0]
importance=pd.DataFrame({"Feature":feature_names,"Coefficient":coef})
importance["Abs importance"]=importance["Coefficient"].abs()
importance=importance.sort_values("Abs importance",ascending=False).head(15)
display(importance)
plt.figure(figsize=(8,5)); sns.barplot(data=importance,x="Coefficient",y="Feature")
plt.title("Model coefficient explainability"); plt.tight_layout(); plt.show()

# Optional SHAP. For a pipeline, explain transformed test data.
try:
    %pip -q install shap
    import shap
    Xt=risk_model.named_steps["prep"].transform(X_test)
    explainer=shap.LinearExplainer(risk_model.named_steps["model"],Xt[:200])
    sv=explainer(Xt[:200])
    shap.summary_plot(sv,features=Xt[:200],feature_names=feature_names,show=True)
except Exception as e:
    print("SHAP unavailable in this environment:",e)

## 15. Confusion matrix, calibration and performance summary

In [ ]:
cm=confusion_matrix(y_test,pred,normalize="true")
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt=".3f",cmap="Blues",
            xticklabels=["No stroke","Stroke"],yticklabels=["No stroke","Stroke"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Normalized confusion matrix"); plt.show()
frac_pos,mean_pred=calibration_curve(y_test,p_test,n_bins=10,strategy="quantile")
plt.figure(figsize=(5,4)); plt.plot(mean_pred,frac_pos,marker="o"); plt.plot([0,1],[0,1],"--")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed fraction"); plt.title("Calibration curve"); plt.show()
summary={"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred,zero_division=0),
         "Recall":recall_score(y_test,pred,zero_division=0),"F1":f1_score(y_test,pred,zero_division=0),
         "ROC_AUC":roc_auc_score(y_test,p_test),"PR_AUC":average_precision_score(y_test,p_test),
         "Brier":brier_score_loss(y_test,p_test)}
summary_df=pd.DataFrame([summary]); display(summary_df)

## 16. Export models, tables, and figures

In [ ]:
# Save model and artifacts
joblib.dump(risk_model,os.path.join(OUTPUT_DIR,"stroke_risk_model.joblib"))
with pd.ExcelWriter(os.path.join(OUTPUT_DIR,"VOFMG_DROC_results.xlsx")) as writer:
    summary_df.to_excel(writer,index=False,sheet_name="Predictive metrics")
    cv_df.to_excel(writer,index=False,sheet_name="5-fold CV")
    ablation_df.to_excel(writer,index=False,sheet_name="Ablation")
    robust_df.to_excel(writer,index=False,sheet_name="Robustness")
    results.to_excel(writer,index=False,sheet_name="Policy simulation")
    importance.to_excel(writer,index=False,sheet_name="Explainability")
    mf_history.to_excel(writer,index=False,sheet_name="Mean-field convergence")
print("Saved to:",OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

## Interpretation and reproducibility checklist

- The predictive classifier is trained/evaluated on the uploaded stroke dataset with a held-out split and stratified 5-fold CV.
- Variable-order memory, treatment response, competing-risk transition, ambiguity scenarios, costs/utilities, and policy simulation are **prototype assumptions** because the manuscript's dataset lacks repeated treatment/outcome trajectories.
- The notebook does not reproduce the manuscript's exact reported percentages by hard-coding them. It calculates outputs from the loaded data and stated assumptions.
- For a clinically defensible full implementation, supply longitudinal patient-level records with timestamps, interventions, stroke events, mortality, adverse effects, costs, and healthcare resource use; then fit transition and treatment-effect models on training patients only.